# Test code

Check the Gurobi license

In [2]:
import gurobipy as gp
print(gp.gurobi.version())

(12, 0, 3)


Import of librabries

In [3]:
import json
from gurobipy import GRB

Import the input

In [41]:
instance = "01"
datasetPath = "../setA"

with open(f"{datasetPath}/setA-{instance}-net.json", "r") as netFile:
    net = json.load(netFile)

with open(f"{datasetPath}/setA-{instance}-scenario.json", "r") as scenarioFile:
    scenario = json.load(scenarioFile)

with open(f"{datasetPath}/setA-{instance}-tm.json", "r") as tmFile:
    tm = json.load(tmFile)

# Network file

V = [node["id"] for node in net["nodes"]]
A = [(link["from"], link["to"]) for link in net["links"]]

# Dict from arc (i, j) to capacity of arc (i, j)
c = {(link["from"], link["to"]): link["capacity"] for link in net["links"]}

# Dict from arc (i, j) to weight/metric of arc (i, j)
weight = {(link["from"], link["to"]): link["metric"] for link in net["links"]}

# Dict from link id to arc (i, j)
link_to_A = {link["id"]: (link["from"], link["to"]) for link in net["links"]}

# Dict from arc (i, j) to link id
A_to_link = {(link["from"], link["to"]): link["id"] for link in net["links"]}


# Scenario file

maxSeg = scenario["max_segments"]

# Dict from time t to budget value at time t
kappla = {i["t"]: i["value"] for i in scenario["budget"]}

# Dict from time t to list of link id (that needs an intervation at time t)
q = {i["t"]: i["links"] for i in scenario["interventions"]}

# Intervation expressed in a arc way instead of the id_link
# Dict from time t to list of arcs (i, j) (that needs an intervation at time t)
q_arc = {t: [link_to_A[link_id] for link_id in links] for t, links in q.items()}

# tm file

num_time_slots = tm["num_time_slots"]
T = list(range(num_time_slots))
T_star = T[1:]
D = list(range(len(tm["demands"])))  # demands indexed 0..n-1

# Dict from demand d to source node s(d)
s = {d: tm["demands"][d]["s"] for d in D}

# Dict from demand d to target node t(d)
t = {d: tm["demands"][d]["t"] for d in D}

# Dict from (d, t) to traffic volume of demand d at time time
v = {(d, time): tm["demands"][d]["v"][time] for d in D for time in T}

MIP model

In [ ]:
def computeSplitCoef()

In [ ]:
m = gp.Model()

pairs = [(i, j) for i in V for j in V if i != j]

# Variable definition

# Binary variable: 1 if the path for demand d uses segment (i,j) at time t
x = m.addVars(D, T, V, V, vtype=GRB.BINARY, name="x")

# Load variable of arc a to time t
lambdaVar = m.addVars(A, T, lb=0.0, name="lambda")

# Buffer variable for the absolute value
y = m.addVars(D, T_star, V, V, lb=0.0, name="y")

# Constraint definition

# (a) Flow conservation constraints
for d in D:
    sd = s[d]
    td = t[d]
    for time in T:
        for i in V:
            buffer = 0
            if i == sd:
                buffer = 1
            elif i == td:
                buffer = -1
            m.addConstr((gp.quicksum(x[d, time, i, j] for j in V if i != j))-gp.quicksum(x[d, time, j, i] for j in V if i != j) == buffer)        
    

# (b) Number of segments
m.addConstrs(gp.quicksum(x[d, time, i, j] for (i, j) in pairs) <= maxSeg for d in D for time in T)

# (c) Load constraints

for time in T:
    ANotInQ = [(i, j) for (i, j) in A if (i, j) not in q_arc.get(time, [])]
    for a in ANotInQ:
        m.addConstr(
            gp.quicksum(r[i, j, a, time] * v[d, time] * x[d, time, i, j] for d in D for (i, j) in pairs)
            <= lambdaVar[a[0], a[1], time] * c[a])

# (d) Budget constraints
m.addConstrs(gp.quicksum(y[d, time, i, j] for d in D for (i,j) in pairs) <= kappla[time] for time in T_star)
m.addConstrs(y[d, time, i, j] >= x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)
m.addConstrs(y[d, time, i, j] >= -x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)

# Objective
m.setObjective(0, GRB.MINIMIZE)
m.optimize()

NameError: name 'r' is not defined

Algorithms and heuristics

In [ ]:
# Objective
m.update()
zlist = []
for a in A:
    for time in T:
        z = m.addVar(lb=0.0)
        tempConstrs = m.addConstrs(lambdaVar[a[0], a[1], time] <= z for a in A for time in T)
        m.setObjective(z, GRB.MINIMIZE)
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            break
        zlist.append(z.X)
        m.addConstrs(lambdaVar[a[0], a[1], time] <= z.X for a in A for time in T)
        m.remove(list(tempConstrs.values()))
        m.remove(z)
        if len(zlist) > 1 and abs(zlist[-1] - zlist[-2]) < 1e-6:
            break

Results and performance analysis